In [ ]:
!pip install pandas numpy plotly -q

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving junction_risk_table.csv to junction_risk_table.csv


In [ ]:
import pandas as pd
import numpy as np

risk_df = pd.read_csv(
    "junction_risk_table.csv"
)

risk_df.head()

,junction,incident_count,high_priority_ratio,road_closure_ratio,planned_event_ratio,unique_causes,cluster,risk_level
0,17th Mn 1st Crs Aishwarya Stores Jn,1,0.000000,0.000000,0.000000,1,3,Medium
1,27th Cross Jayanagar(Ganapathi Temple),1,0.000000,0.000000,0.000000,1,3,Medium
2,28thMainJayanagarJunc,6,0.166667,0.000000,0.000000,1,3,Medium
3,29thMainRdBTM LayoutJunc,5,0.000000,0.000000,0.000000,3,3,Medium
4,5thMainHSR,3,1.000000,0.333333,0.333333,3,2,Low


In [ ]:
cause_score = {

    "vehicle_breakdown":2,

    "pot_holes":2,

    "construction":3,

    "water_logging":3,

    "road_conditions":3,

    "tree_fall":3,

    "accident":4,

    "congestion":4,

    "public_event":5,

    "procession":5,

    "vip_movement":5,

    "protest":5,

    "others":2,

    "debris":2
}

In [ ]:
risk_multiplier = {

    "Low":1,

    "Medium":1.5,

    "High":2,

    "Critical":3
}

In [ ]:
def recommend_resources(
    event_cause,
    priority,
    road_closure,
    risk_level
):

    base_score = cause_score.get(
        event_cause,
        2
    )

    if priority.lower() == "high":
        base_score += 2

    if road_closure:
        base_score += 2

    score = (
        base_score *
        risk_multiplier[risk_level]
    )

    officers = round(
        score * 2
    )

    barricades = round(
        score * 3
    )

    if score < 6:
        response = "Routine"

    elif score < 12:
        response = "Urgent"

    else:
        response = "Critical"

    return {

        "officers":officers,

        "barricades":barricades,

        "response":response
    }

In [ ]:
recommend_resources(

    event_cause="accident",

    priority="High",

    road_closure=True,

    risk_level="Critical"
)

{'officers': 48, 'barricades': 72, 'response': 'Critical'}

In [ ]:
event_cause = input(
    "Event Cause: "
)

priority = input(
    "Priority: "
)

road_closure = input(
    "Road Closure (True/False): "
)

risk_level = input(
    "Risk Level: "
)

result = recommend_resources(

    event_cause,

    priority,

    road_closure=="True",

    risk_level
)

print(result)

Event Cause: accident
Priority: low
Road Closure (True/False): Tue
Risk Level: High
{'officers': 16, 'barricades': 24, 'response': 'Urgent'}


In [ ]:
scenarios = [

    ["vehicle_breakdown",
     "Low",
     False,
     "Low"],

    ["accident",
     "High",
     True,
     "High"],

    ["vip_movement",
     "High",
     True,
     "Critical"],

    ["procession",
     "High",
     True,
     "Critical"]
]

In [ ]:
results = []

for s in scenarios:

    r = recommend_resources(
        s[0],
        s[1],
        s[2],
        s[3]
    )

    results.append({

        "event":s[0],

        "risk":s[3],

        "officers":
        r["officers"],

        "barricades":
        r["barricades"]
    })

pd.DataFrame(results)

,event,risk,officers,barricades
0,vehicle_breakdown,Low,4,6
1,accident,High,32,48
2,vip_movement,Critical,54,81
3,procession,Critical,54,81


In [ ]:
import plotly.express as px

df_plot = pd.DataFrame(results)

fig = px.bar(

    df_plot,

    x="event",

    y="officers",

    color="risk",

    title=
    "Recommended Officers"
)

fig.show()